In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
#################### CHANGE TO MATCH CLASSIFICATION SCRIPT OUPUT ###########################

PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "data" / "speeches.parquet").is_file()
)
FIG_DIR = PROJECT_ROOT / "results" / "figures"
TABLE_DIR = PROJECT_ROOT / "results" / "tables"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(
    PROJECT_ROOT / "data" / "reasoning" / "classification_dataset_topic_style_50k.parquet"
)

In [ ]:
df.head()

In [ ]:
print(df['style_labels'].head())
style_labels = sorted(set(np.concatenate(df["style_labels"].dropna().to_numpy())))
print(style_labels)

In [ ]:
from collections import Counter

import numpy as np
import pandas as pd
from scipy.stats import chi2 as chi2_distribution
from statsmodels.stats.multitest import multipletests


def chi2_keyness(
    df,
    label,
    token_col="tokens",
    labels_col="style_labels",
    min_total_count=50,
    top_n=15,
    smoothing=0.5,
    adjust_pvalues=True,
):
    """
    Calculate chi-square keyness and smoothed log ratio for one label.

    Group p:
        rows where `label` appears in the list stored in `labels_col`

    Reference group:
        rows where `label` does not appear in `labels_col`

    Log ratio:
        log2(P(term | p) / P(term | not p))

    Additive smoothing:
        P(term | p) =
            (count_p + smoothing)
            /
            (total_tokens_p + smoothing * vocabulary_size)

    Parameters
    ----------
    df : pandas.DataFrame
        Input dataframe.

    label : str
        Label defining group p.

    token_col : str, default="tokens"
        Column containing token lists.

    labels_col : str, default="style_labels"
        Column containing lists of labels.

    min_total_count : int, default=50
        Minimum combined count across both corpora.

    top_n : int or None, default=15
        Number of overrepresented terms to return.
        Use None to return all retained terms.

    smoothing : float, default=0.5
        Additive smoothing constant.

    adjust_pvalues : bool, default=True
        Apply Benjamini-Hochberg false-discovery-rate correction.

    Returns
    -------
    pandas.DataFrame
        Terms ranked by chi-square, restricted to terms with positive
        log ratio.
    """

    valid_container_types = (list, tuple, set, np.ndarray)

    def contains_label(value):
        return (
            isinstance(value, valid_container_types)
            and label in value
        )

    def count_tokens(series):
        counts = Counter()

        for tokens in series:
            if isinstance(tokens, valid_container_types):
                counts.update(
                    token
                    for token in tokens
                    if isinstance(token, str) and token
                )

        return counts

    # --------------------------------------------------------
    # Identify target and reference rows
    # --------------------------------------------------------

    group_mask = df[labels_col].map(contains_label)

    n_target_rows = int(group_mask.sum())
    n_reference_rows = int((~group_mask).sum())

    if n_target_rows == 0:
        raise ValueError(
            f"No rows contain the label {label!r}."
        )

    if n_reference_rows == 0:
        raise ValueError(
            f"All rows contain the label {label!r}; "
            "no reference corpus exists."
        )

    # --------------------------------------------------------
    # Count tokens
    # --------------------------------------------------------

    target_counts = count_tokens(
        df.loc[group_mask, token_col]
    )

    reference_counts = count_tokens(
        df.loc[~group_mask, token_col]
    )

    target_total = sum(target_counts.values())
    reference_total = sum(reference_counts.values())

    if target_total == 0:
        raise ValueError(
            "The target corpus contains no valid tokens."
        )

    if reference_total == 0:
        raise ValueError(
            "The reference corpus contains no valid tokens."
        )

    vocabulary = set(target_counts) | set(reference_counts)
    vocabulary_size = len(vocabulary)

    if vocabulary_size == 0:
        raise ValueError("The combined vocabulary is empty.")

    grand_total = target_total + reference_total

    rows = []

    # --------------------------------------------------------
    # Calculate statistics term by term
    # --------------------------------------------------------

    for term in vocabulary:
        count_p = target_counts.get(term, 0)
        count_not_p = reference_counts.get(term, 0)
        total_count = count_p + count_not_p

        if total_count < min_total_count:
            continue

        # 2 x 2 contingency table:
        #
        #                    term      all other tokens
        # target p             a               b
        # reference not p      c               d

        a = count_p
        b = target_total - count_p
        c = count_not_p
        d = reference_total - count_not_p

        non_term_total = grand_total - total_count

        expected_a = target_total * total_count / grand_total
        expected_b = target_total * non_term_total / grand_total
        expected_c = reference_total * total_count / grand_total
        expected_d = reference_total * non_term_total / grand_total

        observed = np.array(
            [a, b, c, d],
            dtype=np.float64,
        )

        expected = np.array(
            [expected_a, expected_b, expected_c, expected_d],
            dtype=np.float64,
        )

        if np.any(expected <= 0):
            continue

        chi2_value = np.sum(
            (observed - expected) ** 2 / expected
        )

        p_value = chi2_distribution.sf(
            chi2_value,
            df=1,
        )

        # Fully normalized additive smoothing
        probability_p = (
            count_p + smoothing
        ) / (
            target_total + smoothing * vocabulary_size
        )

        probability_not_p = (
            count_not_p + smoothing
        ) / (
            reference_total + smoothing * vocabulary_size
        )

        log_ratio = np.log2(
            probability_p / probability_not_p
        )

        rows.append({
            "term": term,
            "count_p": count_p,
            "count_not_p": count_not_p,
            "total_count": total_count,
            "frequency_p_per_million":
                count_p / target_total * 1_000_000,
            "frequency_not_p_per_million":
                count_not_p / reference_total * 1_000_000,
            "chi2": chi2_value,
            "p_value": p_value,
            "log_ratio": log_ratio,
            "frequency_ratio": 2 ** log_ratio,
        })

    result = pd.DataFrame(rows)

    if result.empty:
        return result

    # --------------------------------------------------------
    # Multiple-testing correction
    # --------------------------------------------------------

    if adjust_pvalues:
        result["p_adjusted"] = multipletests(
            result["p_value"],
            method="fdr_bh",
        )[1]

    # Retain words overrepresented in the target corpus
    result = result.loc[
        result["log_ratio"] > 0
    ].copy()

    result = result[
    (result["total_count"] >= 500) &
    (result["p_adjusted"] < 0.01) &
    (result["log_ratio"] > 0)
    ]

    result = (
        result
        .sort_values("log_ratio", ascending=False)
        .head(15)
        .reset_index(drop=True)
    )
    
    if top_n is not None:
        result = result.head(top_n)

    return result.reset_index(drop=True)

In [ ]:
from nltk.tokenize import wordpunct_tokenize

def tokenize_swedish(text):
    if not isinstance(text, str):
        return []

    return [
        token.lower()
        for token in wordpunct_tokenize(text)
        if token.isalpha()
    ]

In [ ]:
df["text_tokens"] = df["text"].map(tokenize_swedish)

In [ ]:
df.head()

In [ ]:
label_terms = {}

for label in style_labels:
    print(label)
    keyness = chi2_keyness(
    df,
    label=label,
    labels_col="style_labels",
    token_col = "text_tokens",
    min_total_count=50,
)
    label_terms[label] = keyness
    

In [ ]:
result_dict = {}
for label, results in label_terms.items():
    print(label)
    print(results['term'])
    result_dict[label] = list(zip(results['term'], results['log_ratio']))
pd.concat(label_terms, names=["style", "rank"]).to_csv(TABLE_DIR / "style_keyness.csv")


In [ ]:
label_name_map = {
    "victim_protection_frame": "Victim-Protection Frame",
    "evidence_statistics_expert_reference": "Evidence-Statistics-Expert Reference",
    "public_safety_security_frame": "Public Safety and Security Frame",
    "legal_institutional_reasoning": "Legal-Institutional Reasoning",
    "rights_based_deontological_appeal": "Rights-Based Deontological Appeal",
    "consequentialist_cost_benefit": "Consequentialist Cost-Benefit",
    "democratic_values_rule_of_law_frame": "Democratic Values and Rule-of-Law Frame",
    "adversarial_blame_attack": "Adversarial Blame/Attack",
    "law_and_order_punitive_frame": "Law-and-Order Punitive Frame",
    "identity_us_them_polarization": "Identity Us-versus-Them Polarization",
    "fairness_equality_justice_frame": "Fairness-Equality-Justice Frame",
    "pragmatic_compromise_consensus": "Pragmatic Compromise/Consensus",
    "moral_polarization_black_white": "Moral Polarization: Black-and-White",
    "market_efficiency_competitiveness_frame": "Market-Efficiency Competitiveness Frame",
    "historical_analogy_ideological_warning": "Historical Analogy/Ideological Warning",
    "humanitarian_compassion_frame": "Humanitarian-Compassion Frame",
    "factual_technical_reporting": "Factual/Technical Reporting",
    "national_sovereignty_frame": "National Sovereignty Frame",
    "formal_ceremonial_address": "Formal Ceremonial Address",
    "preventive_rehabilitation_frame": "Preventive-Rehabilitation Frame",
    "procedural_neutral_descriptive": "Procedural Neutral-Descriptive",
    "anti_authoritarian_moral_condemnation": "Anti-Authoritarian Moral Condemnation",
    "welfare_state_solidarity_frame": "Welfare-State Solidarity Frame",
}

In [ ]:
def escape_latex(value):
    replacements = {
        "&": r"\&",
        "%": r"\%",
        "$": r"\$",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
    }

    value = str(value)

    for old, new in replacements.items():
        value = value.replace(old, new)

    return value


def results_dict_to_latex(
    results_dict,
    label_name_map=label_name_map,
    caption="Top keywords by rhetorical frame",
    table_label="tab:keyness_results",
    top_n=10,
    decimals=2,
):
    """
    Create a multipage LaTeX longtable.

    Each frame contains 10 keywords arranged in two side-by-side
    blocks of five terms.

    Expected results_dict format:
        {
            "frame_label": [
                ("word", log_ratio),
                ...
            ]
        }
    """

    if top_n != 10:
        raise ValueError(
            "This layout is designed for exactly 10 terms per frame."
        )

    sections = []

    for style_label, results in results_dict.items():

        top_results = sorted(
            results,
            key=lambda item: item[1],
            reverse=True,
        )[:top_n]

        # Ensure exactly 10 entries
        top_results += [
            ("", None)
        ] * (top_n - len(top_results))

        clean_label = label_name_map.get(
            style_label,
            style_label,
        )

        rows = []

        # Two blocks:
        # left side = ranks 1--5
        # right side = ranks 6--10
        for i in range(5):
            cells = []

            for block_start in (0, 5):
                idx = block_start + i
                word, score = top_results[idx]

                if score is None:
                    cells.extend(["", "", ""])
                else:
                    cells.extend([
                        str(idx + 1),
                        escape_latex(word),
                        f"{score:.{decimals}f}",
                    ])

            rows.append(
                " & ".join(cells) + r" \\"
            )

        section = rf"""
\addlinespace[0.5em]
\multicolumn{{6}}{{l}}{{\textit{{{escape_latex(clean_label)}}}}} \\[0.2em]
{chr(10).join(rows)}
"""

        sections.append(section.strip())

    latex = rf"""
\begingroup

\footnotesize
\setlength{{\tabcolsep}}{{2.5pt}}
\renewcommand{{\arraystretch}}{{1.08}}

\begin{{longtable}}{{@{{}}r l r @{{\hspace{{0.8em}}}} r l r@{{}}}}

\caption{{{escape_latex(caption)}}}
\label{{{table_label}}} \\

\toprule
Rank & Keyword & LR & Rank & Keyword & LR \\
\midrule
\endfirsthead

\multicolumn{{6}}{{c}}{{%
\tablename\ \thetable\ -- continued from previous page%
}} \\

\toprule
Rank & Keyword & LR & Rank & Keyword & LR \\
\midrule
\endhead

\midrule
\multicolumn{{6}}{{r}}{{Continued on next page}} \\
\endfoot

\bottomrule
\endlastfoot

{chr(10).join(sections)}

\end{{longtable}}

\begin{{minipage}}{{\textwidth}}
\footnotesize
\textit{{Notes:}} LR denotes the base-2 log ratio of normalized term
frequencies. Only terms passing the specified chi-square significance and
frequency thresholds are included. Positive values indicate that a term is
more frequent among speeches assigned to the indicated frame than among
speeches without that frame. A value of 1 indicates twice the normalized
frequency, while a value of 2 indicates four times the normalized frequency.
\end{{minipage}}

\endgroup
"""

    return latex.strip()

In [ ]:
latex_table = results_dict_to_latex(
    result_dict,
    caption="Top keywords associated with rhetorical styles",
    table_label="tab:style_keyness",
)

print(latex_table)

In [ ]:
from matplotlib import pyplot as plt
from pathlib import Path
import math


def plot_keyness_panels(
    results_dict,
    top_n=15,
    ncols=3,
    first_image_rows=2,
    figsize_per_panel=(5, 5),
    save_path=None,
):
    """
    Plot keyness panels across two figures.

    Figure 1 contains the first `first_image_rows` rows of panels.
    Figure 2 contains all remaining panels.

    When `save_path` is provided, files are saved as:
        <filename>_part1.<extension>
        <filename>_part2.<extension>

    Returns
    -------
    tuple
        (first_figure, second_figure)

        `second_figure` is None when there are no remaining panels.
    """
    labels = list(results_dict.keys())
    panels_per_first_image = first_image_rows * ncols

    first_labels = labels[:panels_per_first_image]
    remaining_labels = labels[panels_per_first_image:]

    def create_figure(figure_labels):
        if not figure_labels:
            return None

        n_panels = len(figure_labels)
        nrows = math.ceil(n_panels / ncols)

        fig, axes = plt.subplots(
            nrows,
            ncols,
            figsize=(
                figsize_per_panel[0] * ncols,
                figsize_per_panel[1] * nrows,
            ),
            squeeze=False,
            constrained_layout=True,
        )

        axes = axes.flatten()

        for ax, label in zip(axes, figure_labels):
            results = sorted(
                results_dict[label],
                key=lambda item: item[1],
            )[-top_n:]

            words = [word for word, _ in results]
            scores = [score for _, score in results]
            positions = range(len(words))

            ax.axvline(
                0,
                linewidth=0.8,
                linestyle="--",
            )

            ax.hlines(
                y=positions,
                xmin=0,
                xmax=scores,
                linewidth=1.5,
            )

            ax.scatter(
                scores,
                positions,
                s=35,
                zorder=3,
            )

            max_absolute_score = max(
                (abs(value) for value in scores),
                default=1,
            )
            offset = 0.03 * max(max_absolute_score, 1)

            for y, score in zip(positions, scores):
                x_position = (
                    score + offset
                    if score >= 0
                    else score - offset
                )
                horizontal_alignment = (
                    "left"
                    if score >= 0
                    else "right"
                )

                ax.text(
                    x_position,
                    y,
                    f"{score:.2f}",
                    va="center",
                    ha=horizontal_alignment,
                    fontsize=12,
                )

            clean_label = label_name_map.get(label, label)

            ax.set_title(
                clean_label,
                fontweight="bold",
            )
            ax.set_xlabel(
                "Log-ratio",
                fontsize=12,
            )
            ax.set_yticks(list(positions))
            ax.set_yticklabels(
                words,
                fontsize=12,
            )

            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)

        for ax in axes[n_panels:]:
            ax.remove()

        return fig

    first_fig = create_figure(first_labels)
    second_fig = create_figure(remaining_labels)

    if save_path:
        save_path = Path(save_path)

        first_path = save_path.with_name(
            f"{save_path.stem}_part1{save_path.suffix}"
        )
        first_fig.savefig(
            first_path,
            bbox_inches="tight",
            dpi=300,
        )

        if second_fig is not None:
            second_path = save_path.with_name(
                f"{save_path.stem}_part2{save_path.suffix}"
            )
            second_fig.savefig(
                second_path,
                bbox_inches="tight",
                dpi=300,
            )

    return first_fig, second_fig

In [ ]:
fig1, fig2 = plot_keyness_panels(
    result_dict,
    first_image_rows=4,
    save_path=FIG_DIR / "keyness_panels.png",
)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def plot_average_moral_rating_by_style(
    df,
    labels_col="style_labels",
    rating_col="moral_rating",
    label_name_map=None,
    min_speeches=1,
    figsize=(9, 10),
    save_path=None,
):
    """
    Plot average moral rating for each style label as horizontal bars.

    Rows may contain multiple labels. Each speech contributes to every
    label assigned to it.
    """

    data = df[[labels_col, rating_col]].copy()

    data[rating_col] = pd.to_numeric(
        data[rating_col],
        errors="coerce",
    )

    valid_types = (list, tuple, set, np.ndarray)

    data = data[
        data[labels_col].map(
            lambda value: isinstance(value, valid_types)
        )
    ]

    data = data.explode(labels_col)

    data = data.dropna(
        subset=[labels_col, rating_col]
    )

    summary = (
        data.groupby(labels_col)[rating_col]
        .agg(
            mean_rating="mean",
            n_speeches="size",
        )
        .reset_index()
    )

    summary = summary[
        summary["n_speeches"] >= min_speeches
    ].copy()

    if label_name_map is None:
        label_name_map = {}

    summary["clean_label"] = summary[labels_col].map(
        lambda label: label_name_map.get(label, label)
    )

    summary = summary.sort_values(
        "mean_rating",
        ascending=True,
    ).reset_index(drop=True)

    positions = np.arange(len(summary))

    fig, ax = plt.subplots(figsize=figsize)

    bars = ax.barh(
        positions,
        summary["mean_rating"],
    )

    ax.set_yticks(
        positions,
        labels=summary["clean_label"],
        fontsize=11,
    )

    ax.set_xlabel("Average moral rating")
    ax.set_ylabel("")
    # ax.set_title(
    #     "Average Moral Rating by Rhetorical Frame",
    #     fontweight="bold",
    # )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    value_offset = summary["mean_rating"].max() * 0.01

    for bar, mean in zip(bars, summary["mean_rating"]):
        ax.text(
            mean + value_offset,
            bar.get_y() + bar.get_height() / 2,
            f"{mean:.2f}",
            va="center",
            fontsize=10,
        )

    ax.set_xlim(
        0,
        summary["mean_rating"].max() * 1.10,
    )

   # fig.tight_layout()

    if save_path:
        fig.savefig(
            save_path,
            dpi=300,
            bbox_inches="tight",
        )

    return fig, summary

In [ ]:
fig, moral_rating_summary = plot_average_moral_rating_by_style(
    df,
    labels_col="style_labels",
    rating_col="rating",
    label_name_map=label_name_map,
    min_speeches=1,
    save_path=FIG_DIR / "average_moral_rating_by_frame.png",
)

plt.show()
moral_rating_summary.to_csv(TABLE_DIR / "mean_rating_by_style.csv", index=False)


In [ ]:
len(df)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize


def plot_frame_prevalence_vs_morality(
    df,
    labels_col="style_labels",
    rating_col="moral_rating",
    label_name_map=None,
    min_speeches=100,
    figsize=(10, 11),
    save_path=None,
):
    """
    Plot frame prevalence and average moral rating as two aligned heatmaps.

    The prevalence and moral-rating columns use separate color ranges.
    Each speech contributes to every frame listed in `labels_col`.
    """

    if label_name_map is None:
        label_name_map = {}

    valid_types = (list, tuple, set, np.ndarray)

    data = df[[labels_col, rating_col]].copy()

    data[rating_col] = pd.to_numeric(
        data[rating_col],
        errors="coerce",
    )

    # Keep rows containing list-like labels
    data = data[
        data[labels_col].map(
            lambda value: isinstance(value, valid_types)
        )
    ].copy()

    total_speeches = len(data)

    if total_speeches == 0:
        raise ValueError("No valid label lists were found.")

    # One row per speech-frame combination
    exploded = (
        data.explode(labels_col)
        .dropna(subset=[labels_col, rating_col])
    )

    summary = (
        exploded.groupby(labels_col)
        .agg(
            n_speeches=(rating_col, "size"),
            mean_moral_rating=(rating_col, "mean"),
        )
        .reset_index()
    )

    summary["prevalence_pct"] = (
        100 * summary["n_speeches"] / total_speeches
    )

    summary = summary[
        summary["n_speeches"] >= min_speeches
    ].copy()

    if summary.empty:
        raise ValueError(
            "No frames remain after applying the minimum-speech threshold."
        )

    summary["clean_label"] = summary[labels_col].map(
        lambda label: label_name_map.get(label, label)
    )

    # Highest moral rating at the top
    summary = summary.sort_values(
        "mean_moral_rating",
        ascending=False,
    ).reset_index(drop=True)

    prevalence_values = summary[["prevalence_pct"]].to_numpy()
    morality_values = summary[["mean_moral_rating"]].to_numpy()

    prevalence_norm = Normalize(
        vmin=summary["prevalence_pct"].min(),
        vmax=summary["prevalence_pct"].max(),
    )

    morality_norm = Normalize(
    vmin=1,
    vmax=5,
    )

    fig, (ax_prevalence, ax_morality) = plt.subplots(
        ncols=2,
        figsize=figsize,
        sharey=True,
        gridspec_kw={
            "width_ratios": [1, 1],
            "wspace": 0.2,
        },
    )

    prevalence_image = ax_prevalence.imshow(
        prevalence_values,
        aspect="auto",
        interpolation="nearest",
        norm=prevalence_norm,
        cmap="Blues",
    )

    morality_image = ax_morality.imshow(
        morality_values,
        aspect="auto",
        interpolation="nearest",
        norm=morality_norm,
        cmap="OrRd",
    )

    positions = np.arange(len(summary))

    ax_prevalence.set_yticks(
        positions,
        labels=summary["clean_label"],
        fontsize=12,
    )

    ax_prevalence.set_xticks(
        [0],
        labels=["Prevalence"],
        fontsize=12,
    )

    ax_morality.set_xticks(
        [0],
        labels=["Moral rating"],
        fontsize=12,
    )

    # Actual values inside cells
    for row_index, row in summary.iterrows():
        ax_prevalence.text(
            0,
            row_index,
            f"{row['prevalence_pct']:.1f}%",
            ha="center",
            va="center",
            fontsize=12,
        )

        ax_morality.text(
            0,
            row_index,
            f"{row['mean_moral_rating']:.2f}",
            ha="center",
            va="center",
            fontsize=12,
        )

    ax_prevalence.set_title(
        "Frame commonality",
        fontweight="bold",
        pad=10,
    )

    ax_morality.set_title(
        "Average moralization",
        fontweight="bold",
        pad=10,
    )

    # Remove unnecessary borders
    for ax in (ax_prevalence, ax_morality):
        ax.tick_params(axis="y", length=0)

        for spine in ax.spines.values():
            spine.set_visible(False)

    # Separate colorbars with separate scales
    prevalence_colorbar = fig.colorbar(
        prevalence_image,
        ax=ax_prevalence,
        fraction=0.06,
        pad=0.04,
    )

    prevalence_colorbar.set_label(
        "Speeches with frame (%)",
        rotation=270,
        labelpad=16,
        fontsize=12
    )

    morality_colorbar = fig.colorbar(
        morality_image,
        ax=ax_morality,
        fraction=0.06,
        pad=0.04,
    )

    morality_colorbar.set_label(
        "Average moral rating",
        rotation=270,
        labelpad=16,
        fontsize=12
    )

    # fig.suptitle(
    #     "Commonality and Moralization of Rhetorical Frames",
    #     fontweight="bold",
    #     y=0.995,
    # )

  #  fig.tight_layout()

    if save_path:
        fig.savefig(
            save_path,
            dpi=300,
            bbox_inches="tight",
        )

    return fig, summary

In [ ]:
fig, frame_summary = plot_frame_prevalence_vs_morality(
    df,
    labels_col="style_labels",
    rating_col="rating",
    label_name_map=label_name_map,
    min_speeches=1,
    figsize=(12, 12),
    save_path=FIG_DIR / "frame_prevalence_vs_morality.png",
)

plt.show()
frame_summary.to_csv(TABLE_DIR / "style_prevalence.csv", index=False)


In [ ]:
print(df['rating'].mean())